<a href="https://colab.research.google.com/github/phuc177/Underactuated-robotics/blob/main/book/intro/exercises/drake_systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Run the cells in order in Google Colab. Save a copy in Google Drive to keep your edits.

# Writing (simple) systems in Drake

Drake is a powerful modeling language for authoring models of dynamical systems.  The goal of this exercise is to learn how to author a very simple dynamical systems.  To start, take a few minutes to work through [this Drake tutorial](https://colab.research.google.com/github/RobotLocomotion/drake/blob/master/tutorials/dynamical_systems.ipynb).

Your first task is to write a simple *continuous-time* system that implements the dynamics of a simple damped pendulum (with mass, length, and damping set to 1, and gravity set to 10): $$\ddot\theta + \dot\theta + 10\sin\theta = u.$$  Write code into the following block that results in the pendulum_system variable being an *instance* of a Drake System that implements these dynamics using position and velocity as the state.

Hint: You can import `sin` from `numpy` or from `pydrake.math`. Depending on your version of `numpy`, it might print a warning, but this can be safely ignored.

In [1]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "underactuated[all]>=2026.9.8"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [18]:
import numpy as np
from pydrake.symbolic import Variable
from pydrake.systems.primitives import SymbolicVectorSystem

# Define a new symbolic Variable
theta = Variable("theta")
theta_dot = Variable("theta_dot")
u = Variable("u")
g = 10 # m/s^2
b = 1 # kg.m^2/s
m = 1 # kg
l = 1 # m

# Define the System.
pendulum_system = SymbolicVectorSystem(state=[theta, theta_dot], dynamics=[theta_dot, u-g*np.sin(theta)/l - b*theta_dot/(m*l**2)], output=[theta], input=[u])

Drake [Systems](https://drake.mit.edu/doxygen_cxx/classdrake_1_1systems_1_1_system.html) have methods that define the dynamics, like $$\dot{x} = f(t, x, u), \qquad y = g(t, x, u)$$ for the state dynamics, $f$, and system output function, $g$. Sometimes the equations depend on time, sometimes not. Sometimes they have extra parameters, sometimes not. Not all systems have inputs, and not all systems have states!

To make the input arguments to these methods more succinct, we collect $t, x, u$ and parameters into a single structure, which we call the [Context](https://drake.mit.edu/doxygen_cxx/classdrake_1_1systems_1_1_context.html). Then all of the methods for the dynamical system can be called using that context, e.g.: $$\dot{x} = f(\text{context}), \qquad y = g(\text{context}).$$ We'll use it often throughout the course.

In [10]:
if pendulum_system is not None:
    context = pendulum_system.CreateDefaultContext()
    print(context)

    # Set the time and state
    context.SetTime(4.67)
    context.SetContinuousState([2.1, 3.4])
    print(context)

    # Set an input port to have a fixed value
    # (alternatively, we could connect it to the output port of another system).
    pendulum_system.get_input_port(0).FixValue(context, [7.89])
    print(f"Input port 0 value: {pendulum_system.get_input_port(0).Eval(context)}")

::_ Context
------------
Time: 0
States:
  2 continuous states
    [0, 0]


::_ Context
------------
Time: 4.67
States:
  2 continuous states
    [2.1, 3.4]


Input port 0 value: [7.89]


## Autograding
You can check your work by running the following cell:

In [19]:
from underactuated.exercises.grader import Grader
from underactuated.exercises.intro.test_drake_systems import TestDrakeSystems

Grader.grade_output([TestDrakeSystems], [locals()], "results.json")
Grader.print_test_results("results.json")

Total score is 2/2.

Score for test_dynamics (underactuated.exercises.intro.test_drake_systems.TestDrakeSystems.test_dynamics) is 1/1.

Score for test_input_and_state (underactuated.exercises.intro.test_drake_systems.TestDrakeSystems.test_input_and_state) is 1/1.
